# NB5 - Hierarchical Skills and Strong -> Weak Transfer

**Workshop: Self-Evolving Agents by Optimizing the Harness (no GPU)**

NB4 trained a **flat** skill document: one list, every skill equal, all retrieved
the same way. Two ideas from the skill-optimization literature (SkillX, the
hierarchical-library work) make this scale:

1. **Hierarchy.** Organize skills as **strategies -> tactics**. Retrieve the right
   *strategy* for a question first, then pull only its *tactics*. This is
   coarse-to-fine RAG: fewer, more relevant tokens per prompt.
2. **Strong -> weak transfer.** The *author* of a skill matters. A **strong**
   model (gpt-4o) writes sharper, more general skills than a **weak** one
   (gpt-4o-mini). Inject the strong model's skills into the **weak** agent and the
   weak agent gets better - capability **transferred as text**, no fine-tuning.
   This is how a cheap production model can punch above its weight.

> **Cost note:** this notebook calls **gpt-4o** as the teacher, so it costs a bit
> more than the others (still cents). The *consumer* stays gpt-4o-mini.

In [ ]:
import sys, os, json, re
sys.path.insert(0, os.path.abspath(".."))
from workshop_utils import (
    build_db, load_tasks, run_sql, score_sql, evaluate,
    llm, embed, METER, SCHEMA_TEXT, extract_sql, baseline_prompt, make_agent,
    preflight, flush,
)
preflight("QDRANT_URL", "QDRANT_API_KEY")    # + OPENAI + LANGFUSE (see SETUP.md)
build_db()

STRONG_MODEL = "gpt-4o"                                   # the teacher (authors skills)
WEAK_MODEL   = os.environ.get("WORKSHOP_MODEL", "gpt-4o-mini")  # the consumer

try:
    baseline_acc = json.load(open("../data/baseline_test.json"))["accuracy"]
except FileNotFoundError:
    baseline_acc = evaluate(make_agent(model=WEAK_MODEL), split="test")["accuracy"]
print("weak baseline (no skills) test accuracy:", round(baseline_acc, 3))

## 1. Author skills with a strong vs a weak teacher

We have **gold SQL** for the train tasks, so a teacher can write a skill straight
from each `(question, gold)` pair - no need to run an agent first. We ask each
teacher for a skill plus the **family** (strategy) it belongs to, which gives us
the hierarchy for free. We author from **medium/hard** train tasks (the easy ones
aren't worth a skill).

In [ ]:
AUTHOR_SYS = (
    "You write ONE reusable SKILL for a text-to-SQL agent from a solved example. "
    "Return STRICT JSON with keys: family, name, when_to_use, pattern. "
    "'family' is a short strategy category shared by similar skills "
    "(e.g. completed_revenue, set_difference, grouped_aggregation, date_bucketing). "
    "'when_to_use' triggers a CLASS of questions (no specific values); "
    "'pattern' is the general SQL technique. Generalize - never mention the question."
)

def parse_json_obj(raw):
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

def author_skill(task, model):
    raw = llm([
        {"role": "system", "content": AUTHOR_SYS},
        {"role": "user", "content":
            "Schema:\n" + SCHEMA_TEXT +
            "\nSolved question: " + task["question"] +
            "\nCorrect SQL:\n" + task["gold"] +
            "\n\nReturn the skill as JSON."},
    ], model=model)
    d = parse_json_obj(raw)
    keys = ("family", "name", "when_to_use", "pattern")
    if not d or not all(k in d for k in keys):
        return None
    return {k: str(d[k]) for k in keys}

rich = [t for t in load_tasks()
        if t["split"] == "train" and t["level"] in ("medium", "hard")][:12]

METER.reset()
strong_skills = [s for s in (author_skill(t, STRONG_MODEL) for t in rich) if s]
weak_skills   = [s for s in (author_skill(t, WEAK_MODEL)   for t in rich) if s]
print(f"strong ({STRONG_MODEL}) authored {len(strong_skills)} skills")
print(f"weak   ({WEAK_MODEL}) authored {len(weak_skills)} skills")
print(METER)

## 2. See the hierarchy the strong teacher produced

Grouping the strong model's skills by `family` gives a **strategies -> tactics**
tree. This structure is what we'll retrieve over: pick a strategy, then its tactics.

In [ ]:
from collections import defaultdict
def as_tree(skills):
    fams = defaultdict(list)
    for s in skills:
        fams[s["family"]].append(s)
    return fams

for fam, members in as_tree(strong_skills).items():
    print(f"STRATEGY  {fam}")
    for m in members:
        print(f"    - {m['name']}: USE WHEN {m['when_to_use'][:70]}")

## 3. Index the hierarchy in Qdrant and retrieve coarse-to-fine

We store two kinds of points in one collection: **strategy** points (one per
family) and **tactic** points (one per skill), tagged with `level` and `family`.
Retrieval is two-stage, using Qdrant payload **filters**:
1. find the nearest **strategy** to the question, then
2. find the top tactics **within that family**.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue,
)

COLLECTION = "workshop_skills_hier"
EMBED_DIM = 1536                                   # text-embedding-3-small

qdrant = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

def index_hier(skills, collection=COLLECTION):
    if qdrant.collection_exists(collection):
        qdrant.delete_collection(collection)
    qdrant.create_collection(
        collection, vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE))
    points, pid = [], 0
    for fam, members in as_tree(skills).items():
        desc = "covers: " + ", ".join(m["name"] for m in members)
        points.append(PointStruct(id=pid, vector=embed(f"{fam}. {desc}"),
            payload={"level": "strategy", "family": fam, "description": desc}))
        pid += 1
        vecs = embed([f"{m['name']}. USE WHEN {m['when_to_use']}. {m['pattern']}"
                      for m in members])
        for m, v in zip(members, vecs):
            points.append(PointStruct(id=pid, vector=v,
                payload={"level": "tactic", **m})); pid += 1
    qdrant.upsert(collection, points=points)
    return qdrant.count(collection).count

def _only(level, family=None):
    must = [FieldCondition(key="level", match=MatchValue(value=level))]
    if family is not None:
        must.append(FieldCondition(key="family", match=MatchValue(value=family)))
    return Filter(must=must)

def retrieve_hier(question, k=3, collection=COLLECTION):
    qv = embed(question)
    strat = qdrant.query_points(collection, query=qv, limit=1,
                                query_filter=_only("strategy")).points
    if not strat:
        return []
    fam = strat[0].payload["family"]
    tactics = qdrant.query_points(collection, query=qv, limit=k,
                                  query_filter=_only("tactic", fam)).points
    return strat + tactics

def format_hier(points):
    if not points:
        return ""
    lines = ["Relevant skill hierarchy (apply the strategy, then its tactics):"]
    for p in points:
        d = p.payload
        if d["level"] == "strategy":
            lines.append(f"STRATEGY {d['family']} - {d['description']}")
        else:
            lines.append(f"  - {d['name']}: USE WHEN {d['when_to_use']}. PATTERN: {d['pattern']}")
    return "\n".join(lines)

print("indexed", index_hier(strong_skills), "points (strategies + tactics)")
demo_q = "Which customer has the highest total completed revenue?"
print("\nretrieved for:", demo_q)
print(format_hier(retrieve_hier(demo_q, k=2)))

## 4. The transfer experiment

Same **weak** consumer agent (gpt-4o-mini), same hierarchical retrieval - we only
change **who authored the skills**. If strong -> weak transfer is real, the weak
agent should improve *more* with the strong teacher's skills than with its own.

In [ ]:
def make_consumer(k=3):
    # The weak agent, augmented per-question with retrieved hierarchical skills.
    def agent_fn(question):
        return make_agent(model=WEAK_MODEL,
                          extra=format_hier(retrieve_hier(question, k)))(question)
    return agent_fn

METER.reset()
index_hier(strong_skills)
acc_strong = evaluate(make_consumer(k=3), split="test")["accuracy"]
index_hier(weak_skills)
acc_weak = evaluate(make_consumer(k=3), split="test")["accuracy"]
print("weak baseline (no skills)        :", round(baseline_acc, 3))
print("weak + WEAK-authored skills      :", round(acc_weak, 3))
print("weak + STRONG-authored skills    :", round(acc_strong, 3), " <- transfer")
print(METER)

In [ ]:
import matplotlib.pyplot as plt
labels = ["weak\nbaseline", "weak +\nweak skills", "weak +\nstrong skills"]
vals = [baseline_acc, acc_weak, acc_strong]
plt.figure(figsize=(6, 4))
bars = plt.bar(labels, vals, color=["gray", "steelblue", "seagreen"])
plt.ylim(0, 1); plt.ylabel("test accuracy (weak consumer)")
plt.title("Strong -> weak transfer: better authorship lifts the cheap model")
for b, v in zip(bars, vals):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center")
plt.show()
flush()

## Takeaways

- A **hierarchical** library (strategy -> tactic) retrieves coarse-to-fine, so
  each prompt sees fewer, more on-target skills than a flat dump.
- **Authorship matters.** Skills written by a strong model **transfer** to a weak
  one and lift its accuracy - capability moved as *text*, with the frozen weak
  model still doing inference. This is the cheap-model-punching-up pattern.
- Qdrant payload **filters** turn one collection into a multi-level index - the
  same trick scales to real production skill libraries.

### The gap this leaves (-> NB6)
We still ran each stage **by hand**: author, index, evaluate. NB6 closes the loop -
an **autonomous** agent that evolves its own skill library across generations,
**audits** every change, and reports a final number. The capstone.

### Exercise
1. Sweep `k` (tactics retrieved) from 1 to 5. Where does more context start to
   hurt? Watch the cost meter climb with `k`.
2. Add a third teacher tier (e.g. `gpt-4.1-mini`) and compare the transfer curve.
3. Retrieve the **top-2 strategies** instead of 1 before pulling tactics. Does
   broadening the coarse step help the hard, multi-concept questions?